In [ ]:
!pip install -q faster-whisper fastapi uvicorn pyngrok nest-asyncio transformers bitsandbytes accelerate

In [ ]:
# ── Cell 2: Load models (run once, or after kill_everything()) ────────────────
from faster_whisper import WhisperModel
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

print("Loading Whisper...")
whisper = WhisperModel("large-v3-turbo", device="cuda", compute_type="float16")

MODEL_ID = "Qwen/Qwen2.5-32B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("Loading LLM (this may take a few minutes)...")
llm = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
llm.eval()
print("Models ready.")

In [ ]:
# ── Cell 3: Cleanup — run before reloading models or if you hit OOM ───────────
import gc, subprocess, time, torch

def kill_everything():
    subprocess.run(["pkill", "-9", "-f", "uvicorn"], check=False)
    subprocess.run(["pkill", "-9", "-f", "ngrok"],   check=False)
    time.sleep(2)
    for name in ["llm", "tokenizer", "whisper", "thr"]:
        if name in globals():
            del globals()[name]
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    print("Cleaned up server processes and GPU memory.")

kill_everything()

In [ ]:
# ── Cell 4: FastAPI server ────────────────────────────────────────────────────
import gc
import json
import nest_asyncio
import threading
import tempfile
import os
import uuid
from fastapi.middleware.cors import CORSMiddleware

import torch
import uvicorn
from fastapi import FastAPI, UploadFile, File, BackgroundTasks
from pydantic import BaseModel
from typing import List
from pyngrok import ngrok
from google.colab import userdata

nest_asyncio.apply()

# ── model guards ──────────────────────────────────────────────────────────────
assert "tokenizer" in globals(), "Run Cell 2 first (tokenizer not loaded)"
assert "llm"       in globals(), "Run Cell 2 first (llm not loaded)"

NGROK_AUTH_TOKEN = userdata.get("NGROK_AUTH_TOKEN")

app = FastAPI()
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)
jobs = {}


# ── helpers ───────────────────────────────────────────────────────────────────

def extract_json(text: str):
    text = text.strip()
    if text.startswith("```json"):
        text = text[7:]
    elif text.startswith("```"):
        text = text[3:]
    if text.endswith("```"):
        text = text[:-3]
    return json.loads(text.strip())


MAX_CTX = getattr(llm.config, "max_position_embeddings", 32768)

def run_prompt(prompt: str, max_new_tokens: int | None = None) -> str:
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    input_len = inputs["input_ids"].shape[1]
    budget = MAX_CTX - input_len - 8
    if max_new_tokens is None:
        max_new_tokens = budget

    if max_new_tokens <= 0:
        raise ValueError(f"Prompt is too long! max_new_tokens is {max_new_tokens}.")

    try:
        with torch.no_grad():
            outputs = llm.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        return tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True,
        ).strip()
    finally:
        del inputs
        if "outputs" in locals():
            del outputs

def chunk_text(text: str, max_chars: int = 80000) -> List[str]:
    # Rough estimate: 80,000 chars is ~20,000 tokens, well within 32k context
    chunks = []
    while len(text) > max_chars:
        # Try to split at a newline to avoid cutting sentences in half
        split_idx = text.rfind('\n', 0, max_chars)
        if split_idx == -1:
            split_idx = max_chars
        chunks.append(text[:split_idx])
        text = text[split_idx:].strip()
    if text:
        chunks.append(text)
    return chunks


def summarize(transcript: str) -> dict:
    chunks = chunk_text(transcript)
    summary, attendees = [], []

    for i, chunk in enumerate(chunks):
        print(f"Processing chunk {i+1} of {len(chunks)}...", flush=True)

        summary_prompt = f"""<|im_start|>user
You are analyzing a section (chunk {i+1} of {len(chunks)}) of minutes from a local government meeting. Be thorough and detailed.

Return ONLY a JSON array of ALL agenda items and discussion topics in THIS section, each with:
- "topic": short topic title
- "discussion": 3-5 sentence detailed explanation of what was discussed, including key points raised
- "outcome": the final decision, vote result, or conclusion, or null if none

Rules:
- Do not include routine ceremonial items like the pledge or roll call unless they affected the meeting.
- Capture every topic discussed, even brief ones
- Be specific about vote counts and outcomes
- Do not invent facts.
- Return raw JSON array only, no other keys

Transcript Section:
{chunk}<|im_end|>
<|im_start|>assistant"""

        attendees_prompt = f"""<|im_start|>user
Extract meeting attendees from this section (chunk {i+1} of {len(chunks)}) of the transcript.

Return ONLY a JSON array. Each item must be:
{{
  "name": "full name or null",
  "title": "role/title or null"
}}

Rules:
- Include people who were present in roll call.
- Include people who spoke during the meeting.
- Exclude anyone explicitly marked absent.
- Exclude people only mentioned but not present.
- Deduplicate carefully.
- Use names exactly as stated in the transcript.
- Return raw JSON only.

Transcript Section:
{chunk}
<|im_end|>
<|im_start|>assistant
"""

        try:
            raw = run_prompt(summary_prompt)
            print(f"Summary chunk {i+1} raw:", raw[:200], flush=True)
            chunk_summary = extract_json(raw)
            if isinstance(chunk_summary, list):
                summary.extend(chunk_summary)
        except Exception as e:
            print(f"Summary error on chunk {i+1}: {e}", flush=True)
            summary.append({"topic": f"Processing Error (Chunk {i+1})", "discussion": f"An error occurred during summarization: {str(e)}", "outcome": "Error"})

        try:
            raw = run_prompt(attendees_prompt)
            print(f"Attendees chunk {i+1} raw:", raw[:200], flush=True)
            chunk_attendees = extract_json(raw)
            if isinstance(chunk_attendees, list):
                attendees.extend(chunk_attendees)
        except Exception as e:
            print(f"Attendees error on chunk {i+1}: {e}", flush=True)
            attendees.append({"name": f"Processing Error (Chunk {i+1})", "title": f"An error occurred during attendee extraction: {str(e)}"})

    # Deduplicate attendees after merging chunks
    unique_attendees = []
    seen_names = set()
    for a in attendees:
        name = a.get("name")
        if name and name not in seen_names:
            seen_names.add(name)
            unique_attendees.append(a)

    # --- REDUCE STEP: Summary of Summaries ---
    executive_summary = ""
    if summary:
        print("Generating executive summary from chunked topics...", flush=True)
        reduce_prompt = f"""<|im_start|>user
Below is a JSON list of all topics discussed in a meeting, extracted in sequential chunks.
Write a concise 2-3 paragraph Executive Summary of the entire meeting.
Focus on the most important decisions, major discussions, and the overall outcome.
Return ONLY the summary prose — no JSON, no markdown, no preamble.

Extracted Topics:
{json.dumps(summary, indent=2)}
<|im_end|>
<|im_start|>assistant"""
        try:
            executive_summary = run_prompt(reduce_prompt).strip()
            print(f"Executive summary:", executive_summary[:200], flush=True)
        except Exception as e:
            print(f"Executive summary error: {e}", flush=True)
            executive_summary = f"Error generating executive summary: {str(e)}"

    return {"summary": summary, "attendees": unique_attendees, "executive_summary": executive_summary}

def get_whisper():
  global whisper
  if "whisper" not in globals() or whisper is None:
      whisper = WhisperModel("large-v3-turbo", device="cuda", compute_type="float16")
  return whisper

def process(job_id: str, tmp_path: str):
    try:
        print(f"[{job_id}] Transcribing...", flush=True)
        w = get_whisper()
        segments, _ = w.transcribe(tmp_path, vad_filter=True)
        lines = []
        for segment in segments:
            start = f"{int(segment.start // 60):02d}:{segment.start % 60:05.2f}"
            end   = f"{int(segment.end   // 60):02d}:{segment.end   % 60:05.2f}"
            lines.append(f"[{start} --> {end}] {segment.text.strip()}")
        transcript = "\n".join(lines)
        global whisper
        del whisper
        gc.collect()
        torch.cuda.empty_cache()



        print(f"[{job_id}] Summarizing...", flush=True)
        parsed = summarize(transcript)

        jobs[job_id] = {
            "status":            "done",
            "transcript":        transcript,
            "summary":           parsed.get("summary", []),
            "attendees":         parsed.get("attendees", []),
            "executive_summary": parsed.get("executive_summary", ""),
        }
        print(f"[{job_id}] Done.", flush=True)
    except Exception as e:
        import traceback
        jobs[job_id] = {"status": "error", "message": traceback.format_exc()}
        print(f"[{job_id}] Error: {e}", flush=True)
    finally:
        if os.path.exists(tmp_path):
            os.unlink(tmp_path)

        gc.collect()
        torch.cuda.empty_cache()

def process_resummarize(job_id: str, transcript: str):
    try:
        print(f"[{job_id}] Resummarizing...", flush=True)
        parsed = summarize(transcript)
        jobs[job_id] = {
            "status":            "done",
            "summary":           parsed.get("summary", []),
            "attendees":         parsed.get("attendees", []),
            "executive_summary": parsed.get("executive_summary", ""),
        }
        print(f"[{job_id}] Resummarize Done.", flush=True)
    except Exception as e:
        import traceback
        jobs[job_id] = {"status": "error", "message": traceback.format_exc()}
        print(f"[{job_id}] Error: {e}", flush=True)
    finally:
        gc.collect()
        torch.cuda.empty_cache()

# ── routes ────────────────────────────────────────────────────────────────────

@app.post("/transcribe")
async def transcribe(background_tasks: BackgroundTasks, file: UploadFile = File(...)):
    suffix = os.path.splitext(file.filename or "")[1] or ".tmp"
    with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as tmp:
        tmp.write(await file.read())
        tmp_path = tmp.name
    job_id = str(uuid.uuid4())
    jobs[job_id] = {"status": "processing"}
    background_tasks.add_task(process, job_id, tmp_path)
    print(f"Job {job_id} started", flush=True)
    return {"job_id": job_id}


@app.get("/result/{job_id}")
def result(job_id: str):
    return jobs.get(job_id, {"status": "not_found"})


class ChatMessage(BaseModel):
    role: str
    content: str

class ChatRequest(BaseModel):
    context: str
    messages: List[ChatMessage]


@app.post("/chat")
def chat(req: ChatRequest):
    system = (
        "You are a helpful assistant answering questions about a local government meeting. "
        "Use ONLY the meeting context below. If the answer isn't in the context, say you don't know.\n\n"
        f"MEETING CONTEXT:\n{req.context}"
    )
    prompt = f"<|im_start|>system\n{system}<|im_end|>\n"
    for m in req.messages:
        role = m.role if m.role in ("user", "assistant") else "user"
        prompt += f"<|im_start|>{role}\n{m.content}<|im_end|>\n"
    prompt += "<|im_start|>assistant\n"

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    try:
        with torch.no_grad():
            outputs = llm.generate(
                **inputs,
                max_new_tokens=512,
                temperature=0.3,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
            )
        reply = tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
        ).strip()
        return {"reply": reply}
    finally:
        del inputs
        if "outputs" in locals():
            del outputs
        gc.collect()
        torch.cuda.empty_cache()

class ResummarizeRequest(BaseModel):
    transcript: str

@app.post("/resummarize")
async def resummarize(req: ResummarizeRequest, background_tasks: BackgroundTasks):
    job_id = str(uuid.uuid4())
    jobs[job_id] = {"status": "processing"}
    background_tasks.add_task(process_resummarize, job_id, req.transcript)
    print(f"Resummarize Job {job_id} started", flush=True)
    return {"job_id": job_id}

# ── start server ──────────────────────────────────────────────────────────────

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
public_url = ngrok.connect(8000)
print(f"Ngrok URL: {public_url}", flush=True)

thr = threading.Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000), daemon=True)
thr.start()
